# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanKh4lil/ML-WEEK-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane: CTR / Engagement Opportunity Scoring.** Question this lane asks: *which visible pages
under-capture clicks or engagement for their position, and deserve a metadata/content/monitoring
review?*

This notebook: (1) states the contract in plain words, (2) sorts every field into
feature / label-proxy / context / excluded, (3) proves three facts about the slice with real
queries on a mid-panel month (`month=2026-03`), (4) builds a 5-feature frame with an
"available when?" line each, (5) runs the deliberate-leakage experiment from notebook 02 on
real warehouse data, and (6) names one limitation of this slice.


## 1. The contract, in plain words

**a) What one row means for this lane.**
One row = **one pseudonymized content item (`content_hash_id`), for one client
(`client_hash_id`), on one calendar day (`report_date`)** — the native grain of
`fact_content_daily_performance`. For this lane I aggregate that daily grain UP to
**one row per content item, over a fixed 30-day decision window inside `month=2026-03`**,
because CTR-vs-position judgments are noisy at a single day but stable over ~30 days of
search volume.

**b) Which table(s) I use.**
`fact_content_daily_performance` (the daily fact, partition `month=2026-03`) joined to
`dim_content` (content metadata: `word_count`, `content_created_at`) on `content_hash_id`.
`dim_clients` is used only to sanity-check per-client history depth (`gsc_data_start`), not
as a feature source.

**c) Time window.**
The **decision window** is the 30 days inside `month=2026-03` (2026-03-01 → 2026-03-31,
whatever days that partition actually contains — verified below, since months don't all have
full coverage per client). All 5 features are aggregated over exactly that window. I do not
reach into `month=2026-04` or later for anything, including the label proxy — this is the
mid-panel month, not the sealed final month (`month=2026-06`, the `_sample` table's month).

**d) What I'd predict or rank (label / proxy).**
`is_ctr_review_candidate` — a **defined-rule proxy**, not an observed future outcome:
1 when a content item's 30-day CTR sits well below the median CTR of other items in the
*same position tier* (position-adjusted, so a position-1 page isn't compared to a position-9
page), and the item clears a minimum-impressions floor so the flag isn't noise. This is
exactly Lane 4's "residual/gap analysis" — I never compare raw CTR across different position
tiers.

**e) One thing I deliberately exclude.**
I exclude **`gsc_avg_position` used as a raw continuous feature** — only its bucketed form
(`position_tier`, which I derive myself in section 2) enters the feature frame. Reason: the
dictionary's own warning is that position strata need a volume floor before they mean anything
(the `top_3` stratum can have a median volume of only ~50 impressions/90d in the starter
slice, where a single click swings CTR by ~2pp) — using the raw number invites me to read
precision into a noisy metric that only makes sense once it's bucketed and volume-checked.


In [ ]:
# Setup: DuckDB over the remote Parquet warehouse. No local download.
%pip -q install duckdb
import duckdb

con = duckdb.connect()
# Gated dataset -> register the HF token from Colab Secrets, never pasted in a cell.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import os
    HF_TOKEN = os.environ.get('HF_TOKEN')

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
DIMC = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIMCL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

# Discover the actual column names before assuming anything (contract discipline, not guessing).
print(con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 0").df())


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `content_hash_id`, `client_hash_id` | **Context** | Join/group keys only, pseudonyms, never a feature |
| `report_date` | **Context** | Defines the window boundary, not something the model learns from |
| `gsc_impressions` (30d sum) | **Feature** | Observed search demand, known at the decision moment |
| `gsc_clicks` (30d sum) | Used only to **build the label** (via CTR), never a feature itself |
| `ctr` (= clicks/impressions) | **Label input** | This IS what the proxy label is computed from — never a feature (that would be circular: the model would just relearn its own label) |
| `gsc_avg_position` (30d mean) | **Derived → feature, bucketed** | Raw value excluded (see 1e); `position_tier` (its bucket) is the feature |
| `ga4_sessions` (30d sum) | **Feature** | Observed engagement volume, known at the decision moment |
| `word_count` (`dim_content`) | **Feature** | Static content property, known long before the decision moment |
| `content_created_at` → `content_age_days` | **Feature** | Known at the decision moment (page already existed) |
| `is_ctr_review_candidate` | **Label / proxy** | The thing I rank/predict — a rule-defined stand-in for "deserves a CTR review," not an observed future outcome |
| `ga4_data_available` | **Context / filter** | Availability flag — filtered on, never fed to the model (see section 3c) |

### The five features (max 5, each "knowable at the decision moment because…")

1. **`impressions_30d`** (sum of `gsc_impressions` over the window) — knowable because it is
   search demand *already observed* up through the last day of the decision window.
2. **`sessions_30d`** (sum of `ga4_sessions` over the window) — knowable because GA4 sessions
   in the window are logged as they happen, before any future window exists.
3. **`position_tier`** (bucketed from the window's mean `gsc_avg_position`) — knowable because
   it summarizes where the page already ranked during the window, not where it will rank next.
4. **`word_count`** (from `dim_content`) — knowable because the article's word count was fixed
   at publish/last-edit time, well before this decision window.
5. **`content_age_days`** (decision date − `content_created_at`) — knowable because the page's
   creation date is fixed history, not a future fact.

`ctr` and `gsc_clicks` are deliberately **not** in this list — they're the material the label
proxy is built from, so using them as features would let the model (or even a simple score)
just read its own answer off the input.


In [ ]:
# Build the 5-feature frame for one content-item x 30-day-window grain (month=2026-03).
# Aggregate in SQL first -> only the small result comes back to pandas.
DECISION_WINDOW_START = "DATE '2026-03-01'"
DECISION_WINDOW_END   = "DATE '2026-03-30'"   # verified against the partition's real span in section 3

feature_frame_sql = f"""
WITH windowed AS (
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_impressions)                AS impressions_30d,
        SUM(f.gsc_clicks)                      AS clicks_30d,
        AVG(NULLIF(f.gsc_avg_position, 0))     AS avg_position_30d,
        SUM(f.ga4_sessions)                    AS sessions_30d,
        BOOL_OR(f.ga4_data_available IS TRUE)  AS ga4_available_any
    FROM {{FACT}} f
    WHERE f.report_date BETWEEN {DECISION_WINDOW_START} AND {DECISION_WINDOW_END}
    GROUP BY 1, 2
)
SELECT
    w.content_hash_id,
    w.client_hash_id,
    w.impressions_30d,
    w.clicks_30d,
    ROUND(100.0 * w.clicks_30d / NULLIF(w.impressions_30d, 0), 2) AS ctr,
    w.avg_position_30d,
    CASE
        WHEN w.avg_position_30d IS NULL THEN 'no_data'
        WHEN w.avg_position_30d <= 3  THEN 'top_3'
        WHEN w.avg_position_30d <= 10 THEN 'page_1'
        WHEN w.avg_position_30d <= 20 THEN 'striking'
        WHEN w.avg_position_30d <= 50 THEN 'page_3_5'
        ELSE 'deep'
    END AS position_tier,
    w.sessions_30d,
    w.ga4_available_any,
    d.word_count,
    DATE_DIFF('day', d.content_created_at, {DECISION_WINDOW_END}) AS content_age_days
FROM windowed w
LEFT JOIN {{DIMC}} d ON d.content_hash_id = w.content_hash_id
WHERE w.impressions_30d >= 100   -- volume floor: keep CTR/position comparisons meaningful
""".format(FACT=FACT, DIMC=DIMC)

feature_frame = con.sql(feature_frame_sql).df()
print(feature_frame.shape)
feature_frame.head()


## 3. Verify it with queries (grain, counts, availability)

Three claims from section 1, each proved with a query against `month=2026-03`.


In [ ]:
# (a) GRAIN: one row really is report_date x client_hash_id x content_hash_id.
# Zero rows back means the grain holds.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain rows found:", len(grain_check))
grain_check


In [ ]:
# (b) ROW COUNT + DATE SPAN for this slice's month partition.
span = con.sql(f"""
    SELECT
        COUNT(*)                          AS n_rows,
        MIN(report_date)                  AS min_date,
        MAX(report_date)                  AS max_date,
        COUNT(DISTINCT client_hash_id)    AS n_clients,
        COUNT(DISTINCT content_hash_id)   AS n_content_items
    FROM {FACT}
""").df()
span


In [ ]:
# (c) AVAILABILITY: filter with IS TRUE (never `= FALSE` / plain truthy checks),
# because ga4_data_available can also be NULL, and NULL is neither TRUE nor FALSE.
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_unavailable_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_null_rows
    FROM {FACT}
""").df()
availability['pct_survive_is_true'] = (
    100.0 * availability['ga4_available_rows'] / availability['total_rows']
).round(1)
availability


## 4. Data limits

**Named limitation of this slice:** `month=2026-03` is a mid-panel month, but the panel is
**unbalanced across clients** — some clients' `gsc_data_start` / `ga4_data_start` (in
`dim_clients`) fall after March 2026, so this month simply has no rows for them at all, and
others have only search data with no GA4 rows yet. That means `sessions_30d` and any
feature built from it is systematically thinner for younger clients — not because those pages
have no engagement, but because tracking hadn't started. A model trained on this month alone
will under-weight sessions for exactly the clients with the shortest history, which is a
population that's already underrepresented. This data can never tell me whether a CTR gap is
caused by anything editorial — it's an observational signal, not an experiment.

### The leakage trap (deliberate, then removed)

I train a trivial baseline classifier for `is_ctr_review_candidate` using the 5 honest
features above. Then I add ONE label-derived column on purpose — `ctr` itself, the exact
number the label is computed from — retrain, and watch the score jump toward a perfect score.
Then I delete it and keep the honest number.


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = feature_frame.dropna(subset=['position_tier']).copy()

# Build the proxy label: CTR well below this window's median CTR for the SAME position tier.
tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')
df['is_ctr_review_candidate'] = (df['ctr'] < 0.6 * tier_median_ctr).astype(int)

honest_features = ['impressions_30d', 'sessions_30d', 'word_count', 'content_age_days']
# position_tier is categorical -> one-hot
X_honest = pd.get_dummies(df[honest_features + ['position_tier']], columns=['position_tier'])
y = df['is_ctr_review_candidate']

Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, honest_model.predict_proba(Xte)[:, 1])
print(f"HONEST score (5 features, no leak): ROC AUC = {honest_auc:.3f}")

# --- THE TRAP: add the label-derived column on purpose ---
X_leak = X_honest.copy()
X_leak['ctr'] = df['ctr'].values   # <-- this is literally what the label was computed from

Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_leak, y, test_size=0.3, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(Xtr_l, ytr_l)
leaky_auc = roc_auc_score(yte_l, leaky_model.predict_proba(Xte_l)[:, 1])
print(f"LEAKY score (ctr column added):     ROC AUC = {leaky_auc:.3f}  <-- jumps toward 1.0")

# --- delete the leak, keep the honest number ---
del X_leak
print(f"\nKept number: honest ROC AUC = {honest_auc:.3f}. "
      f"The {leaky_auc - honest_auc:.3f}-point jump came entirely from feeding the model "
      f"the same number its label was derived from -- not from real signal.")


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this in Colab
      with your `HF_TOKEN` secret set; I built this offline and could not execute it against the
      gated Hugging Face warehouse from this environment**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
